##Importing functions

In [0]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.functions import when
from pyspark.sql.functions import min  , max
from pyspark.sql.functions import length,trim
from pyspark.sql.functions import lower,upper
print(' functions imported succesfully')

##reading the table

In [0]:
df=spark.read.table('workspace.bronze.order_details')
print('reading....')

##Bronze Layer Validation

In [0]:
print('Checking schema...')
df.printSchema()
print('Checking Datatypes...')
print(df.dtypes)
print('Checking sample of data...')
df.show(3)


##column operations

In [0]:
#naming convention is snake_case no need to change
#no bad data types but change as defensive approach
df=df.withColumn('order_details_id',col('order_details_id').cast('integer')).withColumn('order_id',col('order_id').cast('integer')).withColumn('pizza_id',col('pizza_id').cast('string')).withColumn('quantity',col('quantity').cast('integer'))
print('Datatypes changed succesfully...')
#check for trailling spaces,trimming
print('trialling spaces',df.filter(length(col("pizza_id")) !=length(trim(col("pizza_id")))).count())
df.withColumn('pizza_id',trim(col("pizza_id")))
print('trimmed as defensive approach...')
#changing pizza_id to lowercase
df=df.withColumn('pizza_id',lower(col("pizza_id")))
print('column pizza_id changed to lowercase...')

#check max,min order_id 
df.select(max('order_id')).show()
df.select(min('order_id')).show()
df.select('order_id').count()

#check max,min pizza_id 
df.select(max('pizza_id')).show()
df.select(min('pizza_id')).show()
df.select('pizza_id').count()


##Row operations

In [0]:
#handle duplicates
##check duplicates
df_unique=df.distinct()
print('unique row numbers:',df_unique.count())
print('original row numbers',df.count())
##drop duplicates as ((defensive approach))
df=df.distinct()
print('dropped duplicates successfully!')
print('-'*40)
#handle nulls 
##check nulls
df_anynulls=df.na.drop('any')
df_allnulls=df.na.drop('all')
if df_anynulls.count() != df.count() or df_allnulls.count() != df.count():
  print('nulls present')
  print('rows count after removing rows containing ANY nulls:',df_anynulls.count())
  print('rows count after removing rows containing ALL nulls: :',df_allnulls.count())
  print('original rows count :',df.count())
  print('action needed')
else:
  print('no nulls present')



##validation

In [0]:
print('-'*40)
print('Data validation checks')
print('-'*40)
validation_results=[]
def validate(colname,rule,condition) :
    invalid= df.filter(condition).count()
    validation_results.append({
        'column' : colname,
        'rule' : rule,
        'invalid_count' : invalid
    })

validate('order_details_id','is negative',col('order_details_id') <= 0)
validate('order_id','is negative',col('order_id') < 0)
validate('quantity','is negative',col('quantity') < 0)
spark.createDataFrame(validation_results).show(truncate=False)
print('-'*40)

#refrential integrity check
df_orders=spark.read.table('workspace.bronze.orders') 
df_pizzas=spark.read.table('workspace.bronze.pizzas')
invalid_order_id= df.join(df_orders,on =df['order_id']==df['order_id'],how='left_anti')
invalid_pizza_id=df.join(df_pizzas,on =df['pizza_id']==df['pizza_id'],how='left_anti')

if invalid_order_id.count() != 0 or invalid_pizza_id.count() != 0:
    print('refrential integrity check failed')
    print('invalid order_id count:',invalid_order_id.count())
    print('invalid pizza_id count :',invalid_pizza_id.count())
    print('action needed')
else :
    print('refrential integrity check passed successfully!')

##Saving

In [0]:
#save as delta 
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.order_details')
print('saved...')


In [0]:
%sql
--SANITY CHECK
SELECT * FROM workspace.silver.order_details LIMIT 10